# 03. Coleta: população (IBGE)

Baixa a população de cada município do Nordeste, ano a ano (2000 a 2026), e grava em `data/raw/ibge/`.

Não depende da coleta da dengue: pode rodar antes, depois ou ao mesmo tempo. As duas fontes só se juntam depois,
pelo código do município.

| Saída | Conteúdo |
|---|---|
| `data/raw/ibge/populacao_municipios_ne.csv` | todos os municípios e anos num arquivo só |
| `data/raw/ibge/{UF}/{município}.csv` | um arquivo por município (ex.: `RN/Natal.csv`) |

**Neste notebook**
1. De onde vêm os dados
2. Coleta
3. Conferência

In [1]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
from coleta.ibge import coleta_ibge as ibge
from config import ANO_INICIO, ANO_FIM_COLETA, N_MUNICIPIOS_NE

## 1. De onde vêm os dados

API SIDRA do IBGE (pública, sem chave). Cada tabela do site https://sidra.ibge.gov.br tem um botão "API" que mostra
o endereço. Exemplo, população de Natal em 2024:
```bash
curl 'https://apisidra.ibge.gov.br/values/t/6579/n6/2408102/v/9324/p/2024'
```

A população de todos os anos precisa de três tabelas, porque os anos de Censo não têm estimativa:

| Tabela | Conteúdo | Anos |
|---|---|---|
| 200 | Censo | 2000 e 2010 |
| 4714 | Censo | 2022 |
| 6579 | Estimativa anual | os demais, exceto 2007 e 2023 |

- Quando um ano tem Censo e estimativa, vale o Censo (é contagem, não projeção).
- **2007 e 2023 não existem** em nenhuma tabela: são preenchidos depois, no `04_carga_ibge.ipynb`.
- Alguns municípios foram criados depois de 2000, por isso os primeiros anos têm um pouco menos de 1794 municípios.

## 2. Coleta
Baixa as três tabelas, junta tudo e grava os arquivos. Leva poucos minutos.
Se o arquivo já existe, não baixa de novo (para baixar outra vez, use `RECOLETAR = True`).

In [2]:
ARQUIVO = ibge.DESTINO / "populacao_municipios_ne.csv"
RECOLETAR = False

if RECOLETAR or not ARQUIVO.exists():
    ibge.main()
else:
    print(f"Já existe: {ARQUIVO}")

Já existe: D:\Users\Rayani\Documents\GitHub\Dengue-eda-northeast-brazil\data\raw\ibge\populacao_municipios_ne.csv


## 3. Conferência
Uma linha por município e ano, com as colunas:

| Coluna | O que é |
|---|---|
| `cod_mun7` | código do município no IBGE (7 dígitos) |
| `cod_mun6` | mesmo código sem o último dígito: é ele que liga com a dengue |
| `uf_cod` | código do estado (21 = MA ... 29 = BA) |
| `municipio` | nome do município |
| `ano` | ano |
| `populacao` | número de habitantes |
| `fonte` | Censo ou estimativa |

In [3]:
pop = pd.read_csv(ARQUIVO, dtype={"cod_mun7": str, "cod_mun6": str})
print(pop.shape)
pop.head()

(44850, 7)


,cod_mun7,cod_mun6,uf_cod,municipio,ano,populacao,fonte
0,2100055,210005,21,Açailândia,2000,88320.0,censo_2000_2010
1,2100055,210005,21,Açailândia,2001,91250.0,estimativa_6579
2,2100055,210005,21,Açailândia,2002,93455.0,estimativa_6579
3,2100055,210005,21,Açailândia,2003,95838.0,estimativa_6579
4,2100055,210005,21,Açailândia,2004,100841.0,estimativa_6579


### 3.1 Municípios por ano
Quantos municípios têm população em cada ano e de qual tabela veio o dado.

In [4]:
cobertura = ibge.cobertura(pop).join(pop.groupby("ano")["fonte"].first())
faltando = sorted(set(range(ANO_INICIO, ANO_FIM_COLETA + 1)) - set(pop["ano"]))
print("Anos sem dado (esperado 2007 e 2023):", faltando)
print(f"Anos com menos de {N_MUNICIPIOS_NE} municípios:", list(cobertura.index[cobertura["municipios"] < N_MUNICIPIOS_NE]))
cobertura

Anos sem dado (esperado 2007 e 2023): [2007, 2023]
Anos com menos de 1794 municípios: [2000, 2001, 2002, 2003, 2004, 2005, 2006]


,municipios,fonte
ano,,
2000,1787,censo_2000_2010
2001,1792,estimativa_6579
2002,1792,estimativa_6579
2003,1792,estimativa_6579
2004,1793,estimativa_6579
2005,1793,estimativa_6579
2006,1793,estimativa_6579
2008,1794,estimativa_6579
2009,1794,estimativa_6579


### 3.2 População por estado
Soma dos municípios de cada estado, para conferir se os números fazem sentido (ex.: RN ~3,3 milhões, BA ~14 milhões).

In [5]:
por_uf = pop.pivot_table(index="ano", columns="uf_cod", values="populacao", aggfunc="sum")
por_uf

uf_cod,21,22,23,24,25,26,27,28,29
ano,,,,,,,,,
2000,5657553.0,2843427.0,7431597.0,2777509.0,3444794.0,7929153.0,2827856.0,1784829.0,13085768.0
2001,5730467.0,2873010.0,7547620.0,2815244.0,3468594.0,8008207.0,2856629.0,1817301.0,13214114.0
2002,5803224.0,2898223.0,7654535.0,2852784.0,3494893.0,8084667.0,2887535.0,1846039.0,13323212.0
2003,5873655.0,2923725.0,7758441.0,2888058.0,3518595.0,8161862.0,2917664.0,1874613.0,13435612.0
2004,6021504.0,2977259.0,7976563.0,2962107.0,3568350.0,8323911.0,2980910.0,1934596.0,13682074.0
2005,6103327.0,3006885.0,8097276.0,3003087.0,3595886.0,8413593.0,3015912.0,1967791.0,13815334.0
2006,6184538.0,3036290.0,8217085.0,3043760.0,3623215.0,8502603.0,3050652.0,2000738.0,13950146.0
2008,6305537.0,3119698.0,8450528.0,3106436.0,3742604.0,8734196.0,3127560.0,1999374.0,14502563.0
2009,6367138.0,3145325.0,8547809.0,3137541.0,3769977.0,8810256.0,3156108.0,2019679.0,14637364.0
